# Quantum Harmonic Oscillator — Physics-Informed Neural Network Solution

## 1. Physical Background

The **quantum harmonic oscillator (QHO)** is the canonical exactly-solvable model of quantum mechanics. Its Hamiltonian operator is:

$$\hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2} + \frac{1}{2}m\omega^2 x^2 \equiv \hat{T} + \hat{V}$$

where $\hat{T}$ is the kinetic energy operator and $\hat{V}$ is the potential energy operator. We solve the **time-independent Schrödinger equation (TISE)**:

$$\hat{H}\psi_n(x) = E_n\psi_n(x), \qquad \psi_n \in L^2(\mathbb{R})$$

### 1.1 Exact Analytic Solution (Natural Units: $\hbar = m = \omega = 1$)

**Eigenvalues:**
$$E_n = \hbar\omega\!\left(n + \tfrac{1}{2}\right) \xrightarrow{\hbar=\omega=1} E_n = n + \tfrac{1}{2}, \quad n = 0, 1, 2, \ldots$$

**Eigenfunctions (Hermite–Gaussian):**
$$\psi_n(x) = \frac{1}{\sqrt{2^n\, n!}}\left(\frac{m\omega}{\pi\hbar}\right)^{1/4} H_n\!\left(\sqrt{\frac{m\omega}{\hbar}}\,x\right) e^{-\frac{m\omega}{2\hbar}x^2}$$

where $H_n(\xi)$ are the **probabilist's Hermite polynomials** satisfying $H_n'' - \xi H_n' + n H_n = 0$:

| $n$ | $H_n(\xi)$ | $\psi_n(x)$ in nat. units |
|-----|-----------|--------------------------|
| 0 | $1$ | $\pi^{-1/4} e^{-x^2/2}$ |
| 1 | $2\xi$ | $\pi^{-1/4}\sqrt{2}\, x\, e^{-x^2/2}$ |
| 2 | $4\xi^2 - 2$ | $\pi^{-1/4}\tfrac{1}{\sqrt{2}}(2x^2-1)e^{-x^2/2}$ |
| 3 | $8\xi^3 - 12\xi$ | $\pi^{-1/4}\tfrac{1}{\sqrt{3}}(2x^3 - 3x)e^{-x^2/2}$ |

### 1.2 Orthonormality and Completeness

The eigenfunctions form a complete orthonormal basis in $L^2(\mathbb{R})$:

$$\langle \psi_m | \psi_n \rangle = \int_{-\infty}^{\infty} \psi_m^*(x)\,\psi_n(x)\,dx = \delta_{mn}$$

$$\sum_{n=0}^{\infty} |\psi_n\rangle\langle\psi_n| = \hat{I}$$

### 1.3 Heisenberg Uncertainty Principle

For eigenstate $|n\rangle$:

$$\sigma_x = \sqrt{\langle x^2\rangle - \langle x\rangle^2} = \sqrt{n + \tfrac{1}{2}}\, \ell_0, \qquad \sigma_p = \sqrt{n+\tfrac{1}{2}}\,\frac{\hbar}{\ell_0}$$

where $\ell_0 = \sqrt{\hbar/(m\omega)}$ is the oscillator length. Thus $\sigma_x \sigma_p = (n+\tfrac{1}{2})\hbar \geq \tfrac{\hbar}{2}$ — saturated for the ground state.

---

## 2. PINN Formulation

### 2.1 Network Ansatz

We approximate the wavefunction by a neural network:

$$\psi_\theta(x) = \mathcal{N}_\theta(x) \cdot \phi(x)$$

where $\phi(x) = e^{-x^2/4}$ is a **Gaussian envelope** that enforces $\psi \to 0$ as $|x| \to \infty$ analytically (hard boundary condition). The network $\mathcal{N}_\theta$ then only needs to learn the polynomial-type correction.

### 2.2 Residual Loss

The PDE residual at collocation point $x_i$:

$$r(x_i;\theta, E) = \hat{H}\psi_\theta(x_i) - E_\theta\,\psi_\theta(x_i) = -\frac{1}{2}\psi_\theta''(x_i) + \frac{x_i^2}{2}\psi_\theta(x_i) - E_\theta\,\psi_\theta(x_i)$$

### 2.3 Composite Loss

$$\mathcal{L}(\theta, E) = \underbrace{\frac{\lambda_{\text{pde}}}{N_c}\sum_{i=1}^{N_c} r(x_i)^2}_{\mathcal{L}_{\text{pde}}} + \underbrace{\frac{\lambda_{\text{bc}}}{2}\!\left[\psi_\theta(-L)^2 + \psi_\theta(+L)^2\right]}_{\mathcal{L}_{\text{bc}}} + \underbrace{\lambda_{\text{norm}}\!\left(\int_{-L}^{L}|\psi_\theta|^2 dx - 1\right)^{\!2}}_{\mathcal{L}_{\text{norm}}}$$

Both $\theta$ and $E_\theta$ are jointly optimized. The second derivative $\psi_\theta''(x)$ is computed **exactly** via PyTorch reverse-mode automatic differentiation — no finite-difference approximation is needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import hermite, factorial
from scipy.integrate import quad

from src.pinn import PINN
from src.physics import harmonic_potential, tise_residual_1d, PINNLoss
from src.data import sample_interior, harmonic_oscillator_ground_state

# ── Reproducibility ────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

plt.style.use('dark_background')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')

# ── Analytic reference functions ───────────────────────────────────────────
def analytic_psi(n: int, x: np.ndarray) -> np.ndarray:
    """Exact QHO eigenfunction for level n in natural units (ℏ=m=ω=1)."""
    norm = 1.0 / np.sqrt(2**n * float(factorial(n))) * np.pi**(-0.25)
    Hn = hermite(n)
    return norm * Hn(x) * np.exp(-x**2 / 2.0)

def analytic_energy(n: int) -> float:
    """Exact energy eigenvalue E_n = n + 0.5 in natural units."""
    return n + 0.5

# ── Demonstration: analytic wavefunctions ─────────────────────────────────
x_plot = np.linspace(-6, 6, 600)
fig, axes = plt.subplots(2, 4, figsize=(16, 6), facecolor='#0d1117')
fig.suptitle('Analytic QHO Eigenstates $\\psi_n(x)$ and Probability Densities $|\\psi_n(x)|^2$',
             color='#e6edf3', fontsize=13)
palette = ['#58a6ff', '#3fb950', '#d2a8ff', '#ffa657']

for n in range(4):
    psi = analytic_psi(n, x_plot)
    V   = 0.5 * x_plot**2
    E   = analytic_energy(n)

    ax_psi  = axes[0, n]
    ax_prob = axes[1, n]

    for ax in (ax_psi, ax_prob):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors='#8b949e', labelsize=8)
        for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

    # Wavefunction (offset to energy level for clarity)
    ax_psi.plot(x_plot, V, color='#30363d', lw=1.5, label='V(x)')
    ax_psi.plot(x_plot, psi + E, color=palette[n], lw=2, label=f'$\\psi_{n}$')
    ax_psi.axhline(E, color=palette[n], lw=0.8, ls='--', alpha=0.5)
    ax_psi.fill_between(x_plot, E, psi + E, alpha=0.15, color=palette[n])
    ax_psi.set_xlim(-6, 6); ax_psi.set_ylim(-0.5, 7)
    ax_psi.set_title(f'$n={n}$,  $E_{n}={E:.1f}$', color='#e6edf3', fontsize=10)
    ax_psi.set_xlabel('$x$', color='#8b949e', fontsize=9)
    ax_psi.set_ylabel('$E + \\psi_n(x)$', color='#8b949e', fontsize=9)

    # Probability density
    ax_prob.plot(x_plot, psi**2, color=palette[n], lw=2)
    ax_prob.fill_between(x_plot, psi**2, alpha=0.3, color=palette[n])
    ax_prob.set_xlim(-6, 6)
    ax_prob.set_xlabel('$x$', color='#8b949e', fontsize=9)
    ax_prob.set_ylabel('$|\\psi_n|^2$', color='#8b949e', fontsize=9)

    # Verify normalization
    norm_val, _ = quad(lambda xi: analytic_psi(n, np.array([xi]))[0]**2, -np.inf, np.inf)
    ax_prob.set_title(f'$\\int|\\psi_{n}|^2 dx = {norm_val:.6f}$', color='#8b949e', fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/qho_analytic_eigenstates.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/qho_analytic_eigenstates.png')

In [ ]:
# ── Hyperparameters & Model Construction ──────────────────────────────────
X_MIN, X_MAX    = -6.0, 6.0
N_COLLOCATION   = 2000
N_BOUNDARY      = 50
N_NORM_QUAD     = 500     # Points for normalization quadrature
N_EPOCHS        = 12000
LR_ADAM         = 1e-3
HIDDEN_DIM      = 80
N_LAYERS        = 5
ACTIVATION      = 'tanh'  # 'tanh', 'sine', or 'gelu'

# Loss weights: higher λ_bc enforces ψ(±L)=0 more strongly
LAM_PDE  = 1.0
LAM_BC   = 100.0
LAM_NORM = 10.0

# ── Gaussian-envelope ansatz  ψ_θ(x) = N_θ(x) · exp(-x²/4) ──────────────
# This imposes the correct L² decay at infinity automatically.
class GaussianEnvelopePINN(nn.Module):
    """PINN with hard Gaussian envelope: ψ(x) = Net(x) * exp(-x²/4).

    The envelope guarantees ψ → 0 as |x| → ∞, converting the soft boundary
    condition penalty into an exactly satisfied constraint.
    """
    def __init__(self, hidden_dim=80, n_layers=5):
        super().__init__()
        layers = []
        dims = [1] + [hidden_dim]*n_layers + [1]
        for i in range(len(dims)-1):
            layers.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                layers.append(nn.Tanh())
        self.net = nn.Sequential(*layers)
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        envelope = torch.exp(-x**2 / 4.0)
        return self.net(x) * envelope

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model  = GaussianEnvelopePINN(hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS).to(device)
energy = torch.tensor([1.5], requires_grad=True, device=device)

optimizer = torch.optim.Adam(list(model.parameters()) + [energy], lr=LR_ADAM)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=1000, factor=0.5, min_lr=1e-6, verbose=False)

print(f'Model parameters  : {model.count_parameters():,}')
print(f'Initial energy E₀ : {energy.item():.3f}')
print(f'Target  energy E₀ : 0.500  (analytic)')
print()
print('Architecture (GaussianEnvelopePINN):')
print('  Input  →  [' + ' → '.join([str(HIDDEN_DIM)]*N_LAYERS) + ']  →  Output')
print('  Activation: tanh')
print('  Envelope   : Net(x) · exp(−x²/4)')

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────
# The loss has three terms:
#   L_pde  = (1/Nc) Σ [H ψ - E ψ]²      ← eigenvalue residual
#   L_bc   = ψ(+L)² + ψ(−L)²            ← Dirichlet BCs (redundant with envelope, kept as extra regularizer)
#   L_norm = (∫ψ² dx − 1)²              ← normalization constraint

history = {'epoch': [], 'total': [], 'pde': [], 'bc': [], 'norm': [], 'energy': []}

dx_quad = (X_MAX - X_MIN) / N_NORM_QUAD  # quadrature step for normalization

bc_x = torch.FloatTensor([[X_MIN], [X_MAX]]).to(device).requires_grad_(True)

for epoch in range(1, N_EPOCHS + 1):
    optimizer.zero_grad()

    # ── Interior collocation points ────────────────────────────────────────
    x_col = torch.FloatTensor(N_COLLOCATION, 1).uniform_(X_MIN, X_MAX).to(device)
    x_col.requires_grad_(True)

    psi_col   = model(x_col)
    dpsi_dx   = torch.autograd.grad(psi_col, x_col,
                                     grad_outputs=torch.ones_like(psi_col),
                                     create_graph=True)[0]
    dpsi_dxx  = torch.autograd.grad(dpsi_dx, x_col,
                                     grad_outputs=torch.ones_like(dpsi_dx),
                                     create_graph=True)[0]

    V_col   = 0.5 * x_col**2                       # V(x) = ½x²
    H_psi   = -0.5 * dpsi_dxx + V_col * psi_col   # Ĥψ
    pde_res = H_psi - energy * psi_col             # residual: Ĥψ − Eψ
    L_pde   = (pde_res**2).mean()

    # ── Boundary condition ─────────────────────────────────────────────────
    psi_bc = model(bc_x)
    L_bc   = (psi_bc**2).mean()

    # ── Normalization via trapezoidal quadrature ───────────────────────────
    x_quad = torch.linspace(X_MIN, X_MAX, N_NORM_QUAD, device=device).unsqueeze(1)
    with torch.no_grad():  # detach for quadrature (no need to differentiate through normalization)
        psi_q = model(x_quad)
    psi_quad = model(x_quad)  # with grad for backprop
    norm_val  = (psi_quad**2 * dx_quad).sum()
    L_norm   = (norm_val - 1.0)**2

    # ── Total loss ─────────────────────────────────────────────────────────
    loss = LAM_PDE * L_pde + LAM_BC * L_bc + LAM_NORM * L_norm

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step(loss.item())

    # ── Logging ────────────────────────────────────────────────────────────
    if epoch % 1000 == 0 or epoch == 1:
        history['epoch'].append(epoch)
        history['total'].append(loss.item())
        history['pde'].append(L_pde.item())
        history['bc'].append(L_bc.item())
        history['norm'].append(L_norm.item())
        history['energy'].append(energy.item())
        print(f'  Epoch {epoch:6d} | L={loss.item():.3e} | '
              f'L_pde={L_pde.item():.3e} | L_norm={L_norm.item():.3e} | '
              f'E={energy.item():.5f} (target 0.50000)')

print(f'\nFinal predicted E₀ = {energy.item():.6f}   |error| = {abs(energy.item()-0.5):.2e}')

# ── Convergence plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

ep = history['epoch']
axes[0].semilogy(ep, history['pde'],   label='$\\mathcal{L}_{\\mathrm{pde}}$',  color='#58a6ff', lw=2)
axes[0].semilogy(ep, history['bc'],    label='$\\mathcal{L}_{\\mathrm{bc}}$',   color='#ffa657', lw=2)
axes[0].semilogy(ep, history['norm'],  label='$\\mathcal{L}_{\\mathrm{norm}}$', color='#d2a8ff', lw=2)
axes[0].semilogy(ep, history['total'], label='$\\mathcal{L}_{\\mathrm{total}}$',color='#e6edf3', lw=2, ls='--')
axes[0].set_xlabel('Epoch', color='#8b949e'); axes[0].set_ylabel('Loss (log)', color='#8b949e')
axes[0].set_title('Training Loss Components', color='#e6edf3')
axes[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')

axes[1].plot(ep, history['energy'], color='#3fb950', lw=2.5, label='Predicted $E_0$')
axes[1].axhline(0.5, color='#ff7b72', lw=1.5, ls='--', label='Analytic $E_0=0.5$')
axes[1].set_xlabel('Epoch', color='#8b949e'); axes[1].set_ylabel('Energy', color='#8b949e')
axes[1].set_title('Eigenvalue Convergence', color='#e6edf3')
axes[1].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')

plt.tight_layout()
plt.savefig('../outputs/qho_training_convergence.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

In [ ]:
# ── Ground-State Evaluation & Visualization ───────────────────────────────
model.eval()
x_test = np.linspace(X_MIN, X_MAX, 600)
psi_ref = analytic_psi(0, x_test)

x_t = torch.FloatTensor(x_test).unsqueeze(1).to(device)
with torch.no_grad():
    psi_pred_raw = model(x_t).cpu().numpy().flatten()

# Fix sign ambiguity (ψ and -ψ are both valid; match analytic sign at x=0)
if np.sign(psi_pred_raw[len(psi_pred_raw)//2]) != np.sign(psi_ref[len(psi_ref)//2]):
    psi_pred_raw = -psi_pred_raw

# Renormalize prediction for fair comparison
dx_test = x_test[1] - x_test[0]
norm_pred = np.sqrt(np.trapz(psi_pred_raw**2, x_test))
psi_pred = psi_pred_raw / norm_pred

# ── Error metrics ──────────────────────────────────────────────────────────
l2_abs   = np.sqrt(np.trapz((psi_pred - psi_ref)**2, x_test))
l2_rel   = l2_abs / np.sqrt(np.trapz(psi_ref**2, x_test))
linf_err = np.max(np.abs(psi_pred - psi_ref))
E_error  = abs(energy.item() - 0.5)

print('━' * 60)
print('  Ground-State Error Summary')
print('━' * 60)
print(f'  Relative L² error     : {l2_rel:.6f}')
print(f'  L² absolute error     : {l2_abs:.6f}')
print(f'  L∞ (max pointwise) err: {linf_err:.6f}')
print(f'  |E_pred − E_exact|    : {E_error:.2e}  '
      f'(E_pred={energy.item():.6f}, E_exact=0.500000)')
print('━' * 60)


In [ ]:
# ── Multi-Panel Comparison ─────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10), facecolor='#0d1117')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35)

def setup_ax(ax):
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

axs = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(3)]
for ax in axs: setup_ax(ax)
palette_l = ['#58a6ff', '#3fb950', '#d2a8ff', '#ffa657']
V_plot = 0.5 * x_test**2

# P1: Wavefunction
axs[0].plot(x_test, psi_ref,  '--', color='#58a6ff', lw=2.5, label='Analytic')
axs[0].plot(x_test, psi_pred, '-',  color='#3fb950', lw=2,   label='PINN')
axs[0].fill_between(x_test, psi_ref, psi_pred, alpha=0.2, color='#ff7b72')
axs[0].set_title('Wavefunction', color='#e6edf3')
axs[0].set_xlabel('$x$', color='#8b949e'); axs[0].set_ylabel('$\\psi_0(x)$', color='#8b949e')
axs[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)

# P2: Probability density
for psi_arr, col, lbl in [(psi_ref**2,'#58a6ff','Analytic'),(psi_pred**2,'#3fb950','PINN')]:
    axs[1].fill_between(x_test, psi_arr, alpha=0.25, color=col)
    axs[1].plot(x_test, psi_arr, color=col, lw=2, label=lbl)
axs[1].set_title('Probability Density $|\\psi|^2$', color='#e6edf3')
axs[1].set_xlabel('$x$', color='#8b949e'); axs[1].set_ylabel('$|\\psi_0|^2$', color='#8b949e')
axs[1].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)

# P3: Pointwise absolute error
abs_err = np.abs(psi_pred - psi_ref)
axs[2].fill_between(x_test, abs_err, alpha=0.5, color='#ff7b72')
axs[2].plot(x_test, abs_err, color='#ff7b72', lw=1.5)
axs[2].axhline(0, color='#8b949e', lw=1, ls='--')
axs[2].set_title(f'Pointwise Error  (max={linf_err:.4f})', color='#e6edf3')
axs[2].set_xlabel('$x$', color='#8b949e'); axs[2].set_ylabel('$|\\psi_\\theta-\\psi_0|$', color='#8b949e')

# P4: Energy level diagram
axs[3].plot(x_test, V_plot, color='#8b949e', lw=1.5)
for n_lvl in range(4):
    En = n_lvl + 0.5
    psi_n = analytic_psi(n_lvl, x_test)
    axs[3].plot(x_test, psi_n * 0.7 + En, color=palette_l[n_lvl], lw=1.5)
    axs[3].axhline(En, color=palette_l[n_lvl], lw=0.8, ls='--', alpha=0.6)
    axs[3].text(5.5, En + 0.05, f'$E_{n_lvl}={En}$', color=palette_l[n_lvl], fontsize=8)
axs[3].set_xlim(-6, 6.5); axs[3].set_ylim(-0.3, 5)
axs[3].set_title('Energy Level Diagram', color='#e6edf3')
axs[3].set_xlabel('$x$', color='#8b949e'); axs[3].set_ylabel('$E+\\psi_n$', color='#8b949e')

# P5: PDE residual profile
x_r = torch.linspace(X_MIN, X_MAX, 300, device=device).unsqueeze(1).requires_grad_(True)
psi_ro = model(x_r)
d1 = torch.autograd.grad(psi_ro, x_r, torch.ones_like(psi_ro), create_graph=True)[0]
d2 = torch.autograd.grad(d1, x_r, torch.ones_like(d1), create_graph=False)[0]
res_np = (-0.5*d2 + 0.5*x_r**2*psi_ro - energy.detach()*psi_ro).detach().cpu().numpy().flatten()
x_r_np = x_r.detach().cpu().numpy().flatten()
axs[4].fill_between(x_r_np, res_np, alpha=0.5, color='#d2a8ff')
axs[4].plot(x_r_np, res_np, color='#d2a8ff', lw=1.5)
axs[4].axhline(0, color='#8b949e', lw=1, ls='--')
axs[4].set_title('PDE Residual $\\hat{H}\\psi_\\theta - E\\psi_\\theta$', color='#e6edf3')
axs[4].set_xlabel('$x$', color='#8b949e'); axs[4].set_ylabel('Residual', color='#8b949e')

# P6: Summary table
axs[5].axis('off')
rows = [['Metric','Value'],
        ['Rel. L² err',  f'{l2_rel:.5f}'],
        ['L∞ err',       f'{linf_err:.5f}'],
        ['Pred. $E_0$',  f'{energy.item():.6f}'],
        ['Exact $E_0$',  '0.500000'],
        ['$|\\Delta E|$',f'{E_error:.2e}'],
        ['Col. pts',     str(N_COLLOCATION)],
        ['Depth / Width',f'{N_LAYERS} / {HIDDEN_DIM}']]
tbl = axs[5].table(cellText=[r for r in rows[1:]], colLabels=rows[0],
                   cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
for (i,j), cell in tbl.get_celld().items():
    cell.set_edgecolor('#30363d')
    cell.set_facecolor('#21262d' if i==0 else '#161b22')
    cell.set_text_props(color='#e6edf3')
axs[5].set_title('Results Summary', color='#e6edf3')

fig.suptitle('QuantumPINN — Ground State Analysis: Quantum Harmonic Oscillator',
             color='#e6edf3', fontsize=13, y=0.99)
plt.savefig('../outputs/qho_ground_state_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 3. Heisenberg Uncertainty Principle Verification

For the $n$-th eigenstate of the QHO, the position and momentum variances are:

$$\langle x \rangle_n = 0, \qquad \langle x^2 \rangle_n = \frac{\hbar}{m\omega}\!\left(n + \tfrac{1}{2}\right)$$

$$\langle p \rangle_n = 0, \qquad \langle p^2 \rangle_n = m\hbar\omega\!\left(n+\tfrac{1}{2}\right)$$

Therefore:

$$\sigma_x \cdot \sigma_p = \left(n + \tfrac{1}{2}\right)\hbar \geq \frac{\hbar}{2}$$

The **ground state saturates** the uncertainty principle ($n = 0 \Rightarrow \sigma_x \sigma_p = \hbar/2$) — it is a **minimum uncertainty state**.

### Numerical Verification via PINN Prediction

We compute $\langle x \rangle$, $\langle x^2 \rangle$, $\langle p^2 \rangle = -\int \psi^* \psi'' dx$ numerically from the PINN output and verify $\sigma_x \sigma_p \approx \hbar/2$.

In [ ]:
# ── Heisenberg Uncertainty Principle Verification ─────────────────────────
# We compute ⟨x⟩, ⟨x²⟩, ⟨p²⟩ = −⟨ψ|∂²/∂x²|ψ⟩ numerically.

def compute_uncertainty(psi_arr, x_arr):
    """Compute σ_x, σ_p, and σ_x·σ_p from a normalized ψ(x) on grid x_arr."""
    dx   = x_arr[1] - x_arr[0]
    norm = np.trapz(psi_arr**2, x_arr)
    psi  = psi_arr / np.sqrt(norm)      # ensure normalized

    x_mean  = np.trapz(psi**2 * x_arr, x_arr)
    x2_mean = np.trapz(psi**2 * x_arr**2, x_arr)
    sigma_x = np.sqrt(x2_mean - x_mean**2)

    # ⟨p²⟩ = ℏ² ⟨ψ|−∂²/∂x²|ψ⟩ = ℏ² ∫ (∂ψ/∂x)² dx  (integration by parts, BC=0)
    dpsidx  = np.gradient(psi, x_arr)  # numerical ∂ψ/∂x
    p2_mean = np.trapz(dpsidx**2, x_arr)   # ℏ=1
    sigma_p = np.sqrt(p2_mean)

    return sigma_x, sigma_p, sigma_x * sigma_p

print('━' * 65)
print(f'  {"State":6s}  {"σ_x":>10s}  {"σ_p":>10s}  {"σ_x·σ_p":>10s}  {"Exact":>10s}  {"Ratio":>8s}')
print('━' * 65)

x_fine = np.linspace(-8, 8, 2000)
results_uncert = {}

for n_lvl in range(4):
    psi_n = analytic_psi(n_lvl, x_fine)
    sx, sp, sxsp = compute_uncertainty(psi_n, x_fine)
    exact = n_lvl + 0.5
    ratio = sxsp / (0.5)          # ratio to ℏ/2 = 0.5
    results_uncert[n_lvl] = (sx, sp, sxsp, exact)
    print(f'  ψ_{n_lvl} (analytic):   σ_x={sx:.5f}  σ_p={sp:.5f}  '
          f'σ_x·σ_p={sxsp:.5f}  exact={(exact):.3f}  ratio={ratio:.4f}')

# Also verify PINN prediction for n=0
sx_pinn, sp_pinn, sxsp_pinn = compute_uncertainty(psi_pred, x_test)
print(f'\n  ψ_0 (PINN pred):   σ_x={sx_pinn:.5f}  σ_p={sp_pinn:.5f}  '
      f'σ_x·σ_p={sxsp_pinn:.5f}  (Exact=0.500)')
print('━' * 65)
print(f'\n  ℏ/2 = 0.5  →  Ground state saturates uncertainty principle: '
      f'σ_x·σ_p = {sxsp_pinn:.5f} ≈ 0.5')

# ── Visualize uncertainty bars ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

states = list(range(4))
sx_vals  = [results_uncert[n][0] for n in states]
sp_vals  = [results_uncert[n][1] for n in states]
sxsp_vals= [results_uncert[n][2] for n in states]
exact_vals = [results_uncert[n][3] for n in states]

axes[0].bar(states, sx_vals, color=['#58a6ff','#3fb950','#d2a8ff','#ffa657'], edgecolor='#30363d')
axes[0].set_xlabel('$n$', color='#8b949e'); axes[0].set_ylabel('$\\sigma_x$', color='#8b949e')
axes[0].set_title('Position Uncertainty $\\sigma_x$', color='#e6edf3')

axes[1].bar(states, sp_vals, color=['#58a6ff','#3fb950','#d2a8ff','#ffa657'], edgecolor='#30363d')
axes[1].set_xlabel('$n$', color='#8b949e'); axes[1].set_ylabel('$\\sigma_p$', color='#8b949e')
axes[1].set_title('Momentum Uncertainty $\\sigma_p$', color='#e6edf3')

axes[2].bar(states, sxsp_vals, color=['#58a6ff','#3fb950','#d2a8ff','#ffa657'], edgecolor='#30363d',
            label='Computed $\\sigma_x \\sigma_p$')
axes[2].plot(states, exact_vals, 'D--', color='#ff7b72', ms=8, lw=2, label='Exact $(n+\\frac{1}{2})$')
axes[2].axhline(0.5, color='#ffa657', lw=1, ls=':', label='$\\hbar/2$')
axes[2].set_xlabel('$n$', color='#8b949e'); axes[2].set_ylabel('$\\sigma_x \\sigma_p$', color='#8b949e')
axes[2].set_title('Uncertainty Product $\\sigma_x \\sigma_p$', color='#e6edf3')
axes[2].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/qho_uncertainty_principle.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/qho_uncertainty_principle.png')

## 4. Excited States via Orthogonality Penalty

The harmonic oscillator eigenstates $\{\psi_n\}$ form a complete orthonormal basis in $L^2(\mathbb{R})$:

$$\langle \psi_m | \psi_n \rangle = \int_{-\infty}^{\infty} \psi_m^*(x)\,\psi_n(x)\,dx = \delta_{mn}$$

To train $\hat\psi_n$ without prior knowledge of Hermite polynomials, we augment the loss with an **orthogonality penalty**. Once the ground state $\hat\psi_0$ is fixed, the first excited state is found by minimizing:

$$\mathcal{L}_{n=1} = \underbrace{\mathcal{L}_{\text{PDE}}[\hat\psi_1]}_{\text{Schrödinger residual}} + \lambda_{\text{orth}} \underbrace{\left(\int_{-\infty}^{\infty} \hat\psi_1(x)\,\hat\psi_0(x)\,dx\right)^2}_{\text{orthogonality to }n=0} + \lambda_{\text{norm}} \underbrace{\left(\int_{-\infty}^{\infty} |\hat\psi_1(x)|^2\,dx - 1\right)^2}_{\text{unit norm}}$$

Similarly, $\hat\psi_2$ is trained with orthogonality penalties against **both** $\hat\psi_0$ and $\hat\psi_1$. This constructs the full eigenspectrum **sequentially**. The TISE residual locks the network to the correct energy level:

$$   \mathcal{L}_{\text{PDE}}[\hat\psi_n] = \frac{1}{N_c}\sum_{i=1}^{N_c}\left[-\frac{\hbar^2}{2m}\hat\psi_n''(x_i) + V(x_i)\hat\psi_n(x_i) - E_n\,\hat\psi_n(x_i)\right]^2 $$

where $E_n = \hbar\omega(n + \tfrac{1}{2})$ is the known exact eigenvalue.

In [ ]:
# ── Train Excited States n = 1, 2, 3 with Orthogonality Penalty ──────────
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import hermite as H_hermite, factorial

# -- Reuse GaussianEnvelopePINN defined in Cell 3 --
# (re-define here for self-containment of this cell)
class GaussianEnvPINN(nn.Module):
    def __init__(self, hidden=64, n_layers=5, activation='tanh'):
        super().__init__()
        act = {'tanh': nn.Tanh(), 'gelu': nn.GELU(), 'silu': nn.SiLU()}[activation]
        layers = [nn.Linear(1, hidden), act]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), act]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x) * torch.exp(-x**2 / 4.0)

def tise_residual(model, x, energy):
    """TISE residual: -½ψ'' + ½x²ψ - E·ψ = 0 (ℏ=m=ω=1)."""
    x = x.requires_grad_(True)
    psi = model(x)
    d1, = torch.autograd.grad(psi.sum(), x, create_graph=True)
    d2, = torch.autograd.grad(d1.sum(), x, create_graph=True)
    V   = 0.5 * x**2
    return -0.5 * d2 + V * psi - energy * psi

def train_excited_state(n_target, prev_models, x_col, x_bc,
                        epochs=3000, lr=1e-3, lambda_orth=20.0, lambda_norm=5.0):
    """Train PINN for the n_target-th excited state."""
    E_exact = n_target + 0.5
    model   = GaussianEnvPINN(hidden=64, n_layers=5)
    energy  = nn.Parameter(torch.tensor([E_exact]))
    opt     = torch.optim.Adam(list(model.parameters()) + [energy], lr=lr)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    history = []

    for ep in range(1, epochs + 1):
        opt.zero_grad()
        # PDE residual
        res = tise_residual(model, x_col, energy)
        L_pde = (res**2).mean()

        # Normalization via trapezoidal rule (differentiable)
        psi_col = model(x_col)
        dx = x_col[1] - x_col[0]
        norm_val = (psi_col**2 * dx).sum()
        L_norm = (norm_val - 1.0)**2

        # Orthogonality penalty against lower states
        L_orth = torch.zeros(1)
        for prev in prev_models:
            prev.eval()
            with torch.no_grad():
                psi_prev = prev(x_col)
            overlap = (psi_col * psi_prev * dx).sum()
            L_orth = L_orth + overlap**2

        loss = L_pde + lambda_norm * L_norm + lambda_orth * L_orth
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        if ep % 500 == 0 or ep == 1:
            history.append((ep, L_pde.item(), L_norm.item(), L_orth.item(), energy.item()))

    return model, history

# ── Setup ────────────────────────────────────────────────────────────────
torch.manual_seed(42)
N_col = 1000
x_col = torch.linspace(-7, 7, N_col).unsqueeze(1)
x_bc  = torch.tensor([[-7.0], [7.0]])

print('Training excited states n = 1, 2, 3  (each ~3000 epochs) ...')
trained_models = [model]       # model from Cell 3 is already the n=0 ground state

table_rows = []
for n_tgt in range(1, 4):
    print(f'  → n = {n_tgt} ...', end=' ', flush=True)
    m_n, hist = train_excited_state(n_tgt, trained_models, x_col, x_bc)
    trained_models.append(m_n)
    final = hist[-1]
    print(f'L_pde={final[1]:.2e}  L_norm={final[2]:.2e}  E={final[4]:.4f}  (exact={n_tgt+0.5})')
    table_rows.append({'n': n_tgt, 'L_pde': final[1], 'E_pinn': final[4], 'E_exact': n_tgt+0.5})

print('\n  n=0 already trained → E_pinn (from Cell 3 energy parameter used here as fixed)')

In [ ]:
# ── Multi-State Comparison: PINN vs Analytic ──────────────────────────────
x_plot = np.linspace(-6, 6, 600)
x_t    = torch.tensor(x_plot, dtype=torch.float32).unsqueeze(1)

fig, axes = plt.subplots(2, 4, figsize=(18, 8), facecolor='#0d1117')
fig.suptitle('QHO Eigenstates: PINN (dashed) vs Analytic (solid)', color='#e6edf3', fontsize=14)
colors = ['#58a6ff', '#3fb950', '#d2a8ff', '#ffa657']

overlap_matrix = np.zeros((4, 4))

for n_idx, (m, col) in enumerate(zip(trained_models, colors)):
    m.eval()
    with torch.no_grad():
        psi_pinn = m(x_t).numpy().flatten()

    psi_analytic = analytic_psi(n_idx, x_plot)

    # Normalize PINN output sign to match analytic
    if np.sign(psi_pinn[300]) != np.sign(psi_analytic[300]):
        psi_pinn = -psi_pinn

    # Normalize ψ
    dx = x_plot[1] - x_plot[0]
    norm = np.trapz(psi_pinn**2, x_plot)
    psi_pinn /= np.sqrt(norm)

    # Wavefunction plot
    ax = axes[0, n_idx]
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    ax.plot(x_plot, psi_analytic, '-', color=col, lw=2.5, label='Analytic')
    ax.plot(x_plot, psi_pinn,     '--', color='#ff7b72', lw=1.5, alpha=0.85, label='PINN')
    ax.fill_between(x_plot, 0, psi_analytic, alpha=0.10, color=col)
    ax.set_title(f'$\\psi_{n_idx}(x)$   $E_{n_idx}={n_idx+0.5}$', color='#e6edf3')
    ax.set_xlabel('$x$', color='#8b949e'); ax.set_ylabel('$\\psi$', color='#8b949e')
    ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)
    ax.axhline(0, color='#30363d', lw=0.8)

    # Probability density plot
    ax2 = axes[1, n_idx]
    ax2.set_facecolor('#161b22'); ax2.tick_params(colors='#8b949e')
    for sp in ax2.spines.values(): sp.set_edgecolor('#30363d')
    ax2.plot(x_plot, psi_analytic**2, '-', color=col, lw=2.5, label='$|\\psi|^2$ analytic')
    ax2.plot(x_plot, psi_pinn**2,     '--', color='#ff7b72', lw=1.5, alpha=0.85, label='$|\\psi|^2$ PINN')
    err = np.sqrt(np.trapz((psi_pinn - psi_analytic)**2, x_plot) /
                  np.trapz(psi_analytic**2, x_plot))
    ax2.set_title(f'$|\\psi_{n_idx}|^2$  rel.L²={err:.4f}', color='#e6edf3')
    ax2.set_xlabel('$x$', color='#8b949e'); ax2.set_ylabel('$|\\psi|^2$', color='#8b949e')
    ax2.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=8)

    # Build overlap matrix
    for m2, col2 in enumerate(trained_models):
        m2.eval()
        with torch.no_grad():
            psi2 = m2(x_t).numpy().flatten()
        n2 = np.trapz(psi2**2, x_plot)
        psi2 /= np.sqrt(n2)
        overlap_matrix[n_idx, m2] = np.trapz(psi_pinn * psi2, x_plot)

plt.tight_layout()
plt.savefig('../outputs/qho_excited_states.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Orthogonality heatmap
fig2, ax3 = plt.subplots(figsize=(5, 4), facecolor='#0d1117')
ax3.set_facecolor('#161b22'); ax3.tick_params(colors='#8b949e')
for sp in ax3.spines.values(): sp.set_edgecolor('#30363d')
im = ax3.imshow(np.abs(overlap_matrix), cmap='RdBu_r', vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax3.text(j, i, f'{overlap_matrix[i,j]:.3f}', ha='center', va='center',
                 color='#e6edf3', fontsize=9)
ax3.set_xticks(range(4)); ax3.set_yticks(range(4))
ax3.set_xticklabels([f'$\\psi_{j}$' for j in range(4)], color='#8b949e')
ax3.set_yticklabels([f'$\\psi_{i}$' for i in range(4)], color='#8b949e')
ax3.set_title('Overlap Matrix $|\\langle\\psi_m|\\psi_n\\rangle|$', color='#e6edf3')
plt.colorbar(im, ax=ax3)
plt.tight_layout()
plt.savefig('../outputs/qho_overlap_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/qho_excited_states.png & qho_overlap_matrix.png')

## 5. Ablation Studies

### 5.1 Collocation Density Ablation

The network's accuracy depends critically on the number of collocation points $N_c$. We define the **relative $L^2$ error** as:

$$\epsilon_{\text{rel}} = \frac{\|\hat\psi - \psi_0\|_2}{\|\psi_0\|_2} = \sqrt{\frac{\int_{-\infty}^{\infty}(\hat\psi(x) - \psi_0(x))^2\,dx}{\int_{-\infty}^{\infty}\psi_0(x)^2\,dx}}$$

As $N_c \to \infty$, the collocation approximation of the integral operators converges, and the error should decrease. Theoretically, for sufficiently smooth networks and PDE data, the error satisfies:

$$\epsilon_{\text{rel}} = \mathcal{O}(N_c^{-\alpha})$$

for some $\alpha > 0$ depending on the activation function and network architecture.

### 5.2 Architecture Depth Ablation

We also vary the number of hidden layers $L \in \{2, 3, 4, 5, 6, 8\}$ and hidden width $W \in \{16, 32, 64, 128\}$ to understand the capacity-accuracy tradeoff. The total parameter count is approximately:

$$N_{\text{params}} \approx W + (L-1)W^2 + W$$

For fixed parameters $N_c = 500$ and fixed training epochs.

In [ ]:
# ── Ablation 1: Collocation Density ──────────────────────────────────────
import time

def quick_train_ground_state(n_col, x_ref, psi_ref, epochs=1500, hidden=32, n_layers=4):
    """Quick training for ablation study."""
    torch.manual_seed(0)
    x_c  = torch.linspace(-6, 6, n_col).unsqueeze(1)
    m    = GaussianEnvPINN(hidden=hidden, n_layers=n_layers)
    E    = nn.Parameter(torch.tensor([0.5]))
    opt  = torch.optim.Adam(list(m.parameters()) + [E], lr=1e-3)
    dx   = x_c[1] - x_c[0]

    for ep in range(epochs):
        opt.zero_grad()
        res = tise_residual(m, x_c, E)
        psi = m(x_c)
        nrm = (psi**2 * dx).sum()
        loss = (res**2).mean() + 5.0 * (nrm - 1)**2
        loss.backward()
        opt.step()

    m.eval()
    with torch.no_grad():
        psi_pred = m(torch.tensor(x_ref, dtype=torch.float32).unsqueeze(1)).numpy().flatten()

    # Fix sign
    if np.sign(psi_pred[len(psi_pred)//2]) < 0:
        psi_pred = -psi_pred

    # Normalize
    dx_ref = x_ref[1] - x_ref[0]
    psi_pred /= np.sqrt(np.trapz(psi_pred**2, x_ref))

    rel_l2 = np.sqrt(np.trapz((psi_pred - psi_ref)**2, x_ref) /
                     np.trapz(psi_ref**2, x_ref))
    return rel_l2

x_ref   = np.linspace(-6, 6, 400)
psi_ref = analytic_psi(0, x_ref)

collocation_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
errors_col        = []

print(f'{"N_col":>8s}  {"Rel.L² error":>14s}')
print('-' * 28)
for nc in collocation_sizes:
    e = quick_train_ground_state(nc, x_ref, psi_ref)
    errors_col.append(e)
    print(f'{nc:>8d}  {e:>14.6f}')

# ── Ablation 2: Architecture Depth ───────────────────────────────────────
depths  = [2, 3, 4, 5, 6, 8]
errors_depth = []
n_params_list  = []

print(f'\n{"Layers":>8s}  {"Rel.L² error":>14s}  {"Params":>8s}')
print('-' * 36)
for nl in depths:
    torch.manual_seed(0)
    m_tmp = GaussianEnvPINN(hidden=32, n_layers=nl)
    np_count = sum(p.numel() for p in m_tmp.parameters())
    e = quick_train_ground_state(500, x_ref, psi_ref, hidden=32, n_layers=nl)
    errors_depth.append(e)
    n_params_list.append(np_count)
    print(f'{nl:>8d}  {e:>14.6f}  {np_count:>8d}')

# ── Plot ablation results ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

axes[0].loglog(collocation_sizes, errors_col, 'o-', color='#58a6ff', lw=2, ms=8, label='Measured')
# Fit slope
log_nc  = np.log(collocation_sizes)
log_err = np.log(errors_col)
slope, intercept = np.polyfit(log_nc[-4:], log_err[-4:], 1)
nc_fit  = np.array([collocation_sizes[0], collocation_sizes[-1]])
axes[0].loglog(nc_fit, np.exp(intercept) * nc_fit**slope, '--', color='#ffa657', lw=1.5,
               label=f'Slope ≈ {slope:.2f}')
axes[0].set_xlabel('$N_c$ (collocation points)', color='#8b949e')
axes[0].set_ylabel('Relative $L^2$ Error', color='#8b949e')
axes[0].set_title('Ablation: Collocation Density', color='#e6edf3')
axes[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
axes[0].grid(True, which='both', color='#21262d', lw=0.5)

axes[1].plot(depths, errors_depth, 's-', color='#3fb950', lw=2, ms=8, label='L² Error')
ax_twin = axes[1].twinx()
ax_twin.plot(depths, n_params_list, '^--', color='#d2a8ff', lw=1.5, ms=7, label='Params')
ax_twin.set_ylabel('Parameter count', color='#d2a8ff')
ax_twin.tick_params(colors='#d2a8ff')
axes[1].set_xlabel('Number of hidden layers', color='#8b949e')
axes[1].set_ylabel('Relative $L^2$ Error', color='#8b949e')
axes[1].set_title('Ablation: Network Depth', color='#e6edf3')
axes[1].legend(loc='upper right', facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
ax_twin.legend(loc='upper left', facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
axes[1].grid(True, color='#21262d', lw=0.5)

plt.tight_layout()
plt.savefig('../outputs/qho_ablation.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Collocation scaling exponent α ≈ {-slope:.3f}')
print('Saved → outputs/qho_ablation.png')

In [ ]:
# ── Activation Function Comparison ────────────────────────────────────────
# Compare tanh, GELU, SiLU for ground-state learning

def quick_train_activation(act_name, n_col=500, epochs=1500):
    torch.manual_seed(42)
    x_c  = torch.linspace(-6, 6, n_col).unsqueeze(1)
    m    = GaussianEnvPINN(hidden=48, n_layers=4, activation=act_name)
    E    = nn.Parameter(torch.tensor([0.5]))
    opt  = torch.optim.Adam(list(m.parameters()) + [E], lr=1e-3)
    dx   = x_c[1] - x_c[0]
    losses = []

    for ep in range(epochs):
        opt.zero_grad()
        res = tise_residual(m, x_c, E)
        psi = m(x_c)
        nrm = (psi**2 * dx).sum()
        loss = (res**2).mean() + 5.0 * (nrm - 1)**2
        loss.backward()
        opt.step()
        if ep % 50 == 0:
            losses.append(loss.item())

    m.eval()
    with torch.no_grad():
        psi_p = m(torch.tensor(x_ref, dtype=torch.float32).unsqueeze(1)).numpy().flatten()
    if np.sign(psi_p[len(psi_p)//2]) < 0:
        psi_p = -psi_p
    dx_r = x_ref[1] - x_ref[0]
    psi_p /= np.sqrt(np.trapz(psi_p**2, x_ref))
    rel_l2 = np.sqrt(np.trapz((psi_p - psi_ref)**2, x_ref) / np.trapz(psi_ref**2, x_ref))
    return losses, rel_l2, psi_p

print('Comparing activation functions: tanh | gelu | silu ...')
act_results = {}
for act in ['tanh', 'gelu', 'silu']:
    lhist, err, psi_a = quick_train_activation(act)
    act_results[act] = {'losses': lhist, 'err': err, 'psi': psi_a}
    print(f'  {act:>6s}:  Rel.L²={err:.6f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')
for ax in axes:
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

act_colors = {'tanh': '#58a6ff', 'gelu': '#3fb950', 'silu': '#ffa657'}
ep_axis    = np.arange(0, 1500, 50)

for act, res in act_results.items():
    axes[0].semilogy(ep_axis[:len(res['losses'])], res['losses'],
                     lw=2, color=act_colors[act], label=f'{act}  ε={res["err"]:.5f}')
    axes[1].plot(x_ref, res['psi'], lw=2, color=act_colors[act], label=act)

axes[1].plot(x_ref, psi_ref, 'w--', lw=1.5, alpha=0.6, label='Analytic')

for ax, ttl in zip(axes, ['Training Loss by Activation', 'Predicted $\\psi_0$ by Activation']):
    ax.set_title(ttl, color='#e6edf3')
    ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3', fontsize=9)

axes[0].set_xlabel('Epoch', color='#8b949e'); axes[0].set_ylabel('Loss', color='#8b949e')
axes[1].set_xlabel('$x$', color='#8b949e'); axes[1].set_ylabel('$\\psi_0(x)$', color='#8b949e')
axes[1].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#e6edf3')
plt.tight_layout()
plt.savefig('../outputs/qho_activation_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → outputs/qho_activation_comparison.png')

In [ ]:
# ── Export All Benchmark Results to CSV ───────────────────────────────────
import pandas as pd

# 1. Collocation ablation
df_col = pd.DataFrame({'n_collocation': collocation_sizes, 'rel_l2_error': errors_col})

# 2. Depth ablation
df_depth = pd.DataFrame({'n_layers': depths, 'n_params': n_params_list, 'rel_l2_error': errors_depth})

# 3. Activation comparison
df_act = pd.DataFrame([
    {'activation': act, 'rel_l2_error': res['err']}
    for act, res in act_results.items()
])

# 4. Multi-state summary
x_eval = np.linspace(-6, 6, 600)
x_te   = torch.tensor(x_eval, dtype=torch.float32).unsqueeze(1)
state_rows = []
for n_idx, m in enumerate(trained_models):
    m.eval()
    with torch.no_grad():
        psi_p = m(x_te).numpy().flatten()
    psi_a = analytic_psi(n_idx, x_eval)
    if np.sign(psi_p[300]) != np.sign(psi_a[300]):
        psi_p = -psi_p
    dx_e = x_eval[1] - x_eval[0]
    psi_p /= np.sqrt(np.trapz(psi_p**2, x_eval))
    rel_l2 = np.sqrt(np.trapz((psi_p - psi_a)**2, x_eval) / np.trapz(psi_a**2, x_eval))
    l_inf  = np.max(np.abs(psi_p - psi_a))
    sx, sp, sxsp = compute_uncertainty(psi_p, x_eval)
    state_rows.append({'n': n_idx, 'E_exact': n_idx + 0.5,
                       'rel_l2': rel_l2, 'l_inf': l_inf,
                       'sigma_x': sx, 'sigma_p': sp, 'sigma_x_sigma_p': sxsp})

df_states = pd.DataFrame(state_rows)

# Save
import os
os.makedirs('../outputs', exist_ok=True)
df_col.to_csv('../outputs/qho_collocation_ablation.csv',  index=False)
df_depth.to_csv('../outputs/qho_depth_ablation.csv',       index=False)
df_act.to_csv('../outputs/qho_activation_ablation.csv',    index=False)
df_states.to_csv('../outputs/qho_full_benchmark.csv',      index=False)

print('Saved CSVs:')
for fn in ['qho_collocation_ablation', 'qho_depth_ablation', 'qho_activation_ablation', 'qho_full_benchmark']:
    print(f'  outputs/{fn}.csv')
print()
print('── Full Benchmark Summary ──────────────────────────────────')
print(df_states.to_string(index=False, float_format='{:.6f}'.format))
print()
print('── Collocation Ablation ────────────────────────────────────')
print(df_col.to_string(index=False, float_format='{:.6f}'.format))
print()
print(f'Power-law fit: ε ∝ N_c^{slope:.3f}  (α≈{-slope:.3f})')